# 02 · Evaluate the prior

Sample `z ~ N(0, I)`, decode, and sanity-check reconstructions of held-out
slices. A good prior produces plausible knee images and reconstructs real
slices closely.

In [ ]:
import jax, jax.numpy as jnp, matplotlib.pyplot as plt
import numpy as np
from mrigen.train_vae import load_model
from mrigen.data import FastMRISlices
from mrigen import viz
vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)
ds = FastMRISlices('../data/processed')

### Prior samples

In [ ]:
key = jax.random.PRNGKey(0)
zs = jax.random.normal(key, (4, vae.latent_dim))
samples = jax.vmap(vae.decoder)(zs)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, s in zip(ax, samples): viz.show_image(s, a)
plt.show()

### Autoencode a real slice  *(needs reparameterise TODO)*

In [ ]:
from mrigen.models.vae import reparameterise
x = jnp.asarray(ds[0])
mu, logvar = vae.encoder(x)
z = reparameterise(mu, logvar, key)
viz.panel(x, vae.decoder(z)); plt.show()

### The prior's own scorecard: quality, diversity, speed

These three numbers are the "prior intrinsics" row of the Friday table, and the fair way to compare
*priors* (VAE vs diffusion vs power spectrum) separately from *reconstructions*.

- **Quality:** do the samples above look like knees? (Eyeball, then the checklist in notebook 06.)
- **Diversity:** mean pairwise (1 − SSIM) over samples — 0 means every sample is the same image.
- **Speed:** seconds per sample; it matters once you compare with a diffusion prior.

In [ ]:
import time
from mrigen import metrics

zs = jax.random.normal(jax.random.PRNGKey(1), (16, vae.latent_dim))
decode_batch = jax.jit(jax.vmap(vae.decoder))
decode_batch(zs).block_until_ready()                      # warm-up (JIT)
t0 = time.perf_counter(); samples = decode_batch(zs).block_until_ready(); dt = time.perf_counter() - t0
print(f"diversity (mean pairwise 1-SSIM over 16 samples): {metrics.diversity(np.asarray(samples)):.3f}")
print(f"speed: {1e3 * dt / len(zs):.2f} ms per sample")

### Latent interpolation

Encode two real slices, walk a straight line between their latent codes, decode. A good prior morphs
one knee into another through *plausible* knees; a bad one passes through mush.

In [ ]:
x0, x1 = jnp.asarray(ds[0]), jnp.asarray(ds[len(ds) // 2])
z0, z1 = vae.encoder(x0)[0], vae.encoder(x1)[0]            # the latent means
ts = jnp.linspace(0, 1, 6)
path = jax.vmap(lambda t: vae.decoder(z0 + t * (z1 - z0)))(ts)
fig, ax = plt.subplots(1, 6, figsize=(18, 3))
for a, img, t in zip(ax, path, ts):
    viz.show_image(img, a, title=f"t = {float(t):.1f}")
plt.show()